# Pipeline metagenómico (versión Google Colab)

Equivalente en Google Colab del notebook `02_pipeline_inicial.ipynb`. Se descargan datos crudos de ENA (https://www.ebi.ac.uk/ena/browser/view/PRJEB11755), se procesan, y se identifican genes y dominios funcionales de la muestra seleccionada. Luego se usa el repositorio público de CARD (https://card.mcmaster.ca/) para identificar los genes de resistencia presentes. El código es para muestras individuales; en el notebook 3 se adapta para analizar varias muestras de forma cíclica.

**Primer borrador**: la lógica del pipeline es idéntica a la versión local; lo único que cambia es cómo se instalan las herramientas de bioinformática (aquí no vienen preinstaladas como en el entorno conda local). Quedan pendientes de ajuste fino en próximas iteraciones.

## 0. Configuración y utilidades

### Entorno en Google Colab

Colab no trae instaladas las herramientas de bioinformática (`fastp`, `seqtk`, `megahit`, `prodigal`, `rgi`) ni conda. Usamos [`condacolab`](https://github.com/conda-incubator/condacolab) para instalar Miniconda y luego `mamba` para instalar los mismos paquetes que en `environment.yml`.

**Importante:** `condacolab.install()` reinicia el runtime automáticamente (verás un mensaje de que la sesión se reinició; es esperado). Después de que reinicie, sigue ejecutando las celdas en orden desde la siguiente.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

El runtime se reinició. **Continúa ejecutando desde la siguiente celda** (no vuelvas a correr la celda anterior).

In [ ]:
import condacolab
condacolab.check()

!mamba install -y -c conda-forge -c bioconda \
    pandas numpy matplotlib biopython \
    fastp seqtk megahit prodigal rgi

### (Opcional) Persistir resultados en Google Drive

Las descargas, el ensamblado (MEGAHIT) y el resto de pasos pueden tardar bastante y el almacenamiento de Colab es efímero (se pierde si se desconecta la sesión). Si quieres conservar `raw/`, `work/` y `results/` entre sesiones, monta tu Drive y ajusta `BASE_DIR` más abajo. Por defecto se deja igual que el notebook original (todo dentro de `/content`).

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# BASE_DIR = Path('/content/drive/MyDrive/amr-ml')
# BASE_DIR.mkdir(parents=True, exist_ok=True)
# os.chdir(BASE_DIR)

In [ ]:
import os, subprocess, hashlib, shutil
from pathlib import Path
import pandas as pd

# ---- Parámetros (edita aquí) ----
RUN       = "ERR1135202"    # run_accession de ENA a procesar
SUBSAMPLE = 1_000_000        # nº de pares a submuestrear (ensayo rápido)
THREADS   = 4                # hilos de CPU
SEED      = 100              # semilla fija -> submuestreo reproducible

# ---- Carpetas de trabajo (se crean si no existen) ----
RAW_DIR  = Path(f"raw/{RUN}")      # FASTQ descargados
WORK_DIR = Path(f"work/{RUN}")     # intermedios
OUT_DIR  = Path(f"results/{RUN}")  # salida final
for d in (RAW_DIR, WORK_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

def sh(cmd):
    """Ejecuta un comando de shell, muestra su salida y aborta si falla."""
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if r.stdout:
        print(r.stdout)
    if r.returncode != 0:
        raise RuntimeError(f"Falló (código {r.returncode}): {cmd}")
    return r.stdout

print("Entorno listo. Muestra a procesar:", RUN)

## 1. Resolver URLs y checksums desde la API de ENA

En vez de adivinar rutas FTP, le preguntamos a ENA por los archivos de este run y sus MD5. `pandas` puede leer la respuesta directamente desde la URL.

In [ ]:
ena_url = (
    "https://www.ebi.ac.uk/ena/portal/api/filereport"
    f"?accession={RUN}&result=read_run"
    "&fields=fastq_ftp,fastq_md5&format=tsv"
)
ena = pd.read_csv(ena_url, sep="\t")
ena

In [ ]:
# Cada campo trae "archivo_1;archivo_2" (paired-end). Los separamos.
ftp = ena.loc[0, "fastq_ftp"].split(";")
md5 = ena.loc[0, "fastq_md5"].split(";")
url1, url2 = "https://" + ftp[0], "https://" + ftp[1]
r1 = RAW_DIR / f"{RUN}_1.fastq.gz"
r2 = RAW_DIR / f"{RUN}_2.fastq.gz"
print(url1)
print(url2)

In [ ]:
# ============================================================
#  Asegurar la base de datos CARD (idempotente y portable)
#  Estrategia en cascada: si ya está cargada la usa; si no,
#  la reutiliza de data/; y si tampoco, la descarga desde cero.
# ============================================================
from pathlib import Path

CARD_URL = "https://card.mcmaster.ca/latest/data"

if Path("localDB/card.json").exists():
    print("CARD ya está cargada en localDB/, no hago nada.")

else:
    if Path("card.json").exists():
        print("Encontré card.json en la raíz; la cargo.")
    elif Path("data/card.json").exists():
        print("Reutilizo la card.json que ya tienes en data/.")
        sh("cp data/card.json card.json")
    else:
        print("No hay CARD en ningún sitio; la descargo (una sola vez)...")
        sh(f"wget -q {CARD_URL} -O card_data.tar.bz2")
        sh("tar -xjf card_data.tar.bz2 ./card.json")

    # Cargar la base en localDB/ del directorio actual
    sh("rgi load --card_json card.json --local")

# Verificación final: si esto falla, algo no cuadró
assert Path("localDB/card.json").exists(), "CARD no quedó cargada; revisa los pasos de arriba."
print("Base CARD lista en:", Path("localDB").resolve())

In [ ]:
# ============================================================
#  Preparar BLAST REMOTO (cero descargas: usa los servidores del NCBI)
#  Solo requiere biopython, que ya está en tu entorno.
# ============================================================
from Bio.Blast import NCBIWWW, NCBIXML
import time, re

# El NCBI pide identificarte para uso remoto (evita bloqueos). Pon tu correo real:
NCBIWWW.email = "jujgomezru@unal.edu.co"

def blast_remoto(fasta_str, program="blastn", db="nt", hitlist=1, max_reintentos=3):
    ultimo = None
    for intento in range(1, max_reintentos + 1):
        try:
            handle = NCBIWWW.qblast(program, db, fasta_str, hitlist_size=hitlist, megablast=True)
            return list(NCBIXML.parse(handle))
        except Exception as e:
            ultimo = e
            print(f"  Intento {intento}/{max_reintentos} -> {type(e).__name__}: {e}")
            if intento < max_reintentos:
                time.sleep(20)
    raise RuntimeError(f"BLAST falló tras {max_reintentos} intentos: "
                       f"{type(ultimo).__name__}: {ultimo}") from ultimo

def organismo_de_descripcion(desc):
    """Extrae 'Género especie' del mejor hit, de forma conservadora."""
    desc = desc.replace("[", "").replace("]", "")
    desc = re.sub(r"^(PREDICTED:|UNVERIFIED:|MAG:|TPA:|TPA_asm:)\s*", "", desc).strip()
    palabras = desc.split()
    if not palabras:
        return None
    if palabras[0].lower() in ("uncultured", "unidentified", "bacterium", "synthetic"):
        return palabras[0].lower()                    # match de baja resolución
    if len(palabras) >= 2 and palabras[1][0].islower():
        return f"{palabras[0]} {palabras[1]}"         # Género especie
    return palabras[0]                                # solo género

print("BLAST remoto listo. Recuerda poner tu correo arriba.")

## 2. Descargar (reanudable) y verificar integridad con MD5

`wget -c` reanuda si la descarga se corta. Después comprobamos el MD5 en Python; si no coincide, abortamos (una descarga corrupta causa errores fantasma más adelante).

In [ ]:
import time, urllib.request, urllib.error

def _human(n):
    for u in ["B", "KB", "MB", "GB"]:
        if n < 1024:
            return f"{n:.1f}{u}"
        n /= 1024
    return f"{n:.1f}TB"

def md5sum(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def download(url, out, want_md5, chunk=1 << 18, max_retries=1000):
    """Descarga robusta: progreso, reanudación tras cortes y verificación MD5."""
    out = Path(out)
    if out.exists() and md5sum(out) == want_md5:
        print(f"Ya existe y el MD5 coincide: {out.name}")
        return
    attempt = 0
    last_print = 0.0
    while True:
        existing = out.stat().st_size if out.exists() else 0     # bytes ya bajados -> reanudar
        headers = {"Range": f"bytes={existing}-"} if existing else {}
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=120) as resp:
                if existing and getattr(resp, "status", 200) == 206:      # el servidor reanuda
                    mode = "ab"
                    total = existing + int(resp.headers.get("Content-Length", 0))
                else:                                                     # empieza de cero
                    existing, mode = 0, "wb"
                    total = int(resp.headers.get("Content-Length", 0))
                done, t0 = existing, time.time()
                with open(out, mode) as f:
                    while True:
                        block = resp.read(chunk)
                        if not block:
                            break
                        f.write(block)
                        done += len(block)
                        now = time.time()
                        if now - last_print >= 0.2 or (total and done >= total):   # refresca ~5 veces/s
                            last_print = now
                            el = now - t0
                            spd = (done - existing) / el if el > 0 else 0
                            pct = (done / total * 100) if total else 0
                            eta = (total - done) / spd if spd > 0 else 0
                            print(f"\r  {out.name}: {pct:5.1f}%  {_human(done)}/{_human(total)}  "
                                  f"{_human(spd)}/s  ETA {int(eta//60)}m{int(eta%60):02d}s   ",
                                  end="", flush=True)
            # CLAVE: si el read terminó pero no llegamos al total, NO está completo -> reanudar
            if total and done < total:
                raise IOError(f"conexión cerrada antes de tiempo: {_human(done)}/{_human(total)}")
            print()
            break
        except urllib.error.HTTPError as e:
            if e.code == 416:                     # el archivo local sobra/está mal -> reiniciar limpio
                print(f"\n  Rango inválido; reinicio {out.name} desde cero.")
                out.unlink(missing_ok=True)
                continue
            raise
        except Exception as e:                    # timeout / cierre / broken pipe -> reintenta y reanuda
            attempt += 1
            if attempt > max_retries:
                raise
            print(f"\n  Interrumpido ({e}); reintentando en 5s [{attempt}]...", flush=True)
            time.sleep(5)
    got = md5sum(out)
    if got != want_md5:
        raise RuntimeError(f"MD5 no coincide para {out.name}: {got} != {want_md5}")
    print(f"  Descarga verificada: {out.name}")

download(url1, r1, md5[0])
download(url2, r2, md5[1])

## 3. Control de calidad y recorte de adaptadores (fastp)

Genera además un reporte visual `.html` que puedes abrir para ver el antes/después.

In [ ]:
clean1 = WORK_DIR / "clean_1.fastq.gz"
clean2 = WORK_DIR / "clean_2.fastq.gz"
if clean1.exists() and clean2.exists():
    print("QC ya realizado, se omite")
else:
    sh(f"fastp -i {r1} -I {r2} -o {clean1} -O {clean2} "
       f"-w {THREADS} -h {OUT_DIR}/{RUN}_fastp.html -j {OUT_DIR}/{RUN}_fastp.json")

## 4. Submuestreo reproducible (seqtk)

Tomamos una fracción al azar con la **misma semilla** en ambos pares, para que las lecturas sigan emparejadas. Esto hace el ensayo rápido; para la muestra completa, sube `SUBSAMPLE` o sáltate este paso.

In [ ]:
sub1 = WORK_DIR / "sub_1.fastq"
sub2 = WORK_DIR / "sub_2.fastq"
if sub1.exists() and sub2.exists():
    print("Submuestreo ya realizado, se omite")
else:
    sh(f"seqtk sample -s{SEED} {clean1} {SUBSAMPLE} > {sub1}")
    sh(f"seqtk sample -s{SEED} {clean2} {SUBSAMPLE} > {sub2}")

## 5. Ensamblado (MEGAHIT)

Reconstruye contigs a partir de las lecturas. **Este paso es el más lento** (varios minutos); en el notebook su salida aparecerá cuando termine, no en vivo. Ten paciencia.

In [ ]:
import subprocess, time

def sh_live(cmd):
    """Ejecuta un comando mostrando su salida EN VIVO, con el tiempo transcurrido."""
    print(f"$ {cmd}\n")
    t0 = time.time()
    proc = subprocess.Popen(cmd, shell=True, text=True, bufsize=1,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        el = time.time() - t0
        print(f"[{int(el//60):>2}m{int(el%60):02d}s] {line}", end="", flush=True)
    proc.wait()
    el = time.time() - t0
    if proc.returncode != 0:
        raise RuntimeError(f"Falló (código {proc.returncode}) tras {int(el)}s: {cmd}")
    print(f"\nCompletado en {int(el//60)}m{int(el%60):02d}s")

contigs = WORK_DIR / "megahit_out" / "final.contigs.fa"
if contigs.exists():
    print("Ensamblado ya existe, se omite")
else:
    shutil.rmtree(WORK_DIR / "megahit_out", ignore_errors=True)  # MEGAHIT exige que NO exista
    sh_live(f"megahit -1 {sub1} -2 {sub2} -t {THREADS} -o {WORK_DIR}/megahit_out")

In [ ]:
import pandas as pd
from Bio import SeqIO

# IDs reales de tus contigs (del ensamblado que ya existe)
asm_ids = {r.id for r in SeqIO.parse(str(contigs), "fasta")}
print(f"Contigs en el ensamblado: {len(asm_ids)}")   # sanity: debe ser > 0

rgi = pd.read_csv(f"{rgi_out}.txt", sep="\t")

def contig_de_orf(orf):
    """Deduce el contig real del ORF_ID de RGI, anclándose en los IDs del ensamblado."""
    orf = str(orf).split()[0]
    if orf in asm_ids:
        return orf
    partes = orf.split("_")
    for corte in range(len(partes) - 1, 0, -1):
        cand = "_".join(partes[:corte])
        if cand in asm_ids:
            return cand
    return None

rgi["contig_asm"] = rgi["ORF_ID"].map(contig_de_orf)
matched = rgi["contig_asm"].notna().sum()
print(f"Filas de RGI emparejadas a un contig real: {matched}/{len(rgi)}")

wanted = set(rgi["contig_asm"].dropna())
recs = [r for r in SeqIO.parse(str(contigs), "fasta") if r.id in wanted]
print(f"Contigs con ARG extraídos: {len(recs)}")

In [ ]:
import pandas as pd

filas = []
for i, r in enumerate(recs, 1):
    print(f"[{i}/{len(recs)}] BLAST de {r.id} ...", flush=True)
    try:
        registros = blast_remoto(f">{r.id}\n{r.seq}\n")   # un contig por envío
        rec = registros[0]
        if rec.alignments:
            top = rec.alignments[0]; hsp = top.hsps[0]
            pid = 100 * hsp.identities / hsp.align_length
            filas.append({"contig": r.id,
                          "taxon": organismo_de_descripcion(top.hit_def),
                          "identidad_%": round(pid, 1),
                          "descripcion": top.hit_def[:70]})
        else:
            filas.append({"contig": r.id, "taxon": None, "identidad_%": None, "descripcion": "sin hits"})
    except Exception as e:
        print(f"    Falló {r.id}: {e}")
        filas.append({"contig": r.id, "taxon": None, "identidad_%": None, "descripcion": f"error: {e}"})

blast_df = pd.DataFrame(filas)
contig_to_species = dict(zip(blast_df["contig"], blast_df["taxon"]))
blast_df

In [ ]:
import time, urllib.request, urllib.error, threading
from pathlib import Path

def _human(n):
    for u in ["B", "KB", "MB", "GB", "TB"]:
        if n < 1024:
            return f"{n:.1f}{u}"
        n /= 1024
    return f"{n:.1f}PB"

def descargar(url, out, chunk=1 << 15, timeout=30, max_retries=1000):
    """Descarga con LATIDO en hilo aparte: siempre muestra fase, tiempo y bytes, aunque esté esperando."""
    out = Path(out)
    st = {"done": 0, "total": 0, "fase": "conectando", "t0": time.time(), "vivo": True}

    def latido():
        while st["vivo"]:
            time.sleep(1)
            if not st["vivo"]:
                break
            el = int(time.time() - st["t0"])
            if st["total"]:
                pct = st["done"] / st["total"] * 100
                print(f"\r  [{el:>3}s] {st['fase']}: {pct:4.1f}%  {_human(st['done'])}/{_human(st['total'])}     ", end="", flush=True)
            else:
                print(f"\r  [{el:>3}s] {st['fase']}: {_human(st['done'])} descargado     ", end="", flush=True)

    hb = threading.Thread(target=latido, daemon=True); hb.start()
    attempt = 0
    try:
        while True:
            existing = out.stat().st_size if out.exists() else 0
            headers = {"Range": f"bytes={existing}-"} if existing else {}
            st["fase"] = "conectando"; st["t0"] = time.time()
            try:
                with urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=timeout) as resp:
                    if existing and getattr(resp, "status", 200) == 206:
                        mode = "ab"; total = existing + int(resp.headers.get("Content-Length", 0))
                    else:
                        existing, mode = 0, "wb"; total = int(resp.headers.get("Content-Length", 0))
                    st["total"] = total; st["done"] = existing; st["fase"] = "descargando"; done = existing
                    with open(out, mode) as f:
                        while True:
                            block = resp.read1(chunk)
                            if not block:
                                break
                            f.write(block); done += len(block); st["done"] = done
                if total and done < total:
                    raise IOError(f"conexión cerrada antes de tiempo: {_human(done)}/{_human(total)}")
                st["vivo"] = False
                print(f"\n  Descarga completa: {out.name} ({_human(out.stat().st_size)})")
                return
            except urllib.error.HTTPError as e:
                if e.code == 416:
                    st["vivo"] = False; print(f"\n  {out.name} ya está completo."); return
                raise
            except Exception as e:
                attempt += 1
                if attempt > max_retries:
                    raise
                print(f"\n  Interrumpido ({type(e).__name__}: {e}); reintentando [{attempt}]...", flush=True)
                time.sleep(2)
    finally:
        st["vivo"] = False

# --- Descargar la anotación de taxonomía del catálogo ---
BASE   = "https://ftp.cngb.org/pub/SciRAID/Microbiome/pig/"
NR_TAX = BASE + "GeneAnnotation/pigGut.GeneCatalog.NR-tax.annotation.gz"

Path("catalog").mkdir(exist_ok=True)
descargar(NR_TAX, "catalog/pig_NR_tax.annotation.gz")

In [ ]:
import gzip, pandas as pd, re
from collections import Counter

path = "catalog/pig_NR_tax.annotation.gz"

# 1) Estructura cruda: ¿hay encabezado?, ¿cómo separa?
print("=== Primeras líneas ===")
with gzip.open(path, "rt") as f:
    for i, line in enumerate(f):
        print(line.rstrip()[:180])
        if i >= 3:
            break

# 2) Columnas
peek = pd.read_csv(path, sep="\t", compression="gzip", nrows=5)
print("\nColumnas:", peek.columns.tolist())

# 3) Detectar la columna de especie (evitando la de tax_id)
col_tax = next((c for c in peek.columns
                if re.search(r"tax|species|scientific", str(c), re.I)
                and "id" not in str(c).lower()), None)
print("Columna de taxonomía:", col_tax)

if col_tax:
    # 4) Censo sobre TODO el archivo, por trozos (son millones de genes)
    conteo = Counter()
    for chunk in pd.read_csv(path, sep="\t", compression="gzip",
                             usecols=[col_tax], chunksize=500_000):
        conteo.update(chunk[col_tax].dropna().astype(str))
    print(f"\nGenes con taxonomía asignada: {sum(conteo.values()):,}")
    print(f"Taxones distintos (tu censo): {len(conteo):,}")

    # 5) ¿Cubre los géneros que esperas por el resistoma de tu muestra?
    esperados = ["Bacteroides", "Prevotella", "Clostridium", "Escherichia", "Lactobacillus"]
    print("\n¿Están los géneros esperados de tu muestra?")
    for g in esperados:
        hits = sum(v for k, v in conteo.items() if g.lower() in k.lower())
        print(f"  {g:14s}: {'sí' if hits else 'no'} ({hits:,} genes)")

    # 6) Los 15 taxones más abundantes en el catálogo
    print("\nTop 15 taxones del catálogo:")
    for tax, n in conteo.most_common(15):
        print(f"  {n:>8,}  {tax}")
else:
    print("\nNo detecté la columna de taxonomía automáticamente; pégame la lista de columnas y la ajusto.")

## 6. Predicción de genes (Prodigal)

Encuentra los genes dentro de los contigs y los guarda como proteínas (`.faa`) y nucleótidos (`.fna`).

In [ ]:
genes_faa = WORK_DIR / "genes.faa"
genes_fna = WORK_DIR / "genes.fna"
if not genes_faa.exists():
    sh(f"prodigal -i {contigs} -a {genes_faa} -d {genes_fna} -p meta -q")

n_genes = int(subprocess.run(f"grep -c '>' {genes_faa}", shell=True,
                             text=True, capture_output=True).stdout or 0)
print(f"Genes predichos: {n_genes}")

## 7. Detección de genes de resistencia (RGI contra CARD)

Usamos las proteínas (`-t protein`) porque ya las predijo Prodigal. Requiere la base CARD cargada localmente.

In [ ]:
if not (Path("localDB/card.json").exists() or Path("card.json").exists()):
    raise RuntimeError(
        "No encuentro la base CARD local. Corre antes en esta carpeta:\n"
        "  rgi load --card_json card.json --local"
    )

rgi_out = OUT_DIR / f"rgi_{RUN}"
sh(f"rgi main -i {genes_faa} -o {rgi_out} -t protein -a DIAMOND --local --clean")

## 8. Cargar y visualizar el resistoma

Aquí es donde el notebook brilla: leemos la tabla de RGI con pandas y la resumimos. Estas columnas son las que alimentarán después el modelo (familia de ARG, clase de antibiótico, mecanismo, y el identificador ARO).

In [ ]:
rgi = pd.read_csv(f"{rgi_out}.txt", sep="\t")
print(f"Genes de resistencia detectados: {len(rgi)}")
print("Por tipo de acierto (Cut_Off):")
print(rgi["Cut_Off"].value_counts())

rgi[["Best_Hit_ARO", "Drug Class", "Resistance Mechanism", "AMR Gene Family"]]

In [ ]:
import matplotlib.pyplot as plt

fam = rgi["AMR Gene Family"].value_counts()
ax = fam.plot(kind="barh")
ax.invert_yaxis()
plt.xlabel("Nº de genes detectados")
plt.title(f"Familias de genes de resistencia — {RUN}")
plt.tight_layout()
plt.show()

---
**Siguiente paso.** Este resistoma es el lado de la *etiqueta* para el modelo (qué familias de ARG hay). Lo que falta es el *quién*: a qué bacteria pertenece cada gen (asignación taxonómica), para luego cruzarlo con los rasgos de BacDive. Ese es el puente hacia la tabla de entrenamiento.

In [ ]:
import pandas as pd

# 1) Tu tabla de RGI real (la que ya generaste)
rgi = pd.read_csv(f"{rgi_out}.txt", sep="\t")[["Contig", "Best_Hit_ARO", "Drug Class", "AMR Gene Family"]]

# 2) La columna PUENTE: contig -> especie (esto lo produce el paso taxonómico que falta).
#    De momento lo dejo como un diccionario de ejemplo para que veas el encadenado;
#    lo reemplazarás por el resultado real de clasificar tus contigs con ARG.
contig_to_species = {
    "k99_123": "Prevotella copri",
    "k99_456": "Bacteroides fragilis",
    # ... una entrada por cada contig con ARG
}
rgi["taxon"] = rgi["Contig"].map(contig_to_species)

# 3) Consultar BacDive UNA vez por cada especie distinta (no por cada gen) y unir
especies = rgi["taxon"].dropna().unique()
rasgos = pd.DataFrame([bacdive_traits(sp) for sp in especies])   # tu función de antes

tabla_final = rgi.merge(rasgos, on="taxon", how="left")
tabla_final